# Ex1

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "facebook/nllb-200-distilled-600M"

tok = AutoTokenizer.from_pretrained(model_id, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

src = "Yesterday, I visited a small bookstore near my apartment. The owner recommended a novel about a young musician who moves to Paris. Although I usually prefer science fiction, I decided to give it a try. I read the first three chapters while waiting for the bus. By the time I got home, I was eager to find out what happened next."
inputs = tok(src, return_tensors="pt")

out = model.generate(
    **inputs, 
    forced_bos_token_id=tok.convert_tokens_to_ids("fra_Latn"), 
    num_beams=5, 
    length_penalty=1.0,
)
output = tok.batch_decode(out, skip_special_tokens=True)[0]
new_src = output




Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

## Round Trip

In [3]:
tok2 = AutoTokenizer.from_pretrained(model_id, src_lang="fra_Latn")
inputs2 = tok2(new_src, return_tensors="pt")
out = model.generate(
    **inputs2, 
    forced_bos_token_id=tok.convert_tokens_to_ids("eng_Latn"), 
    num_beams=5, 
    length_penalty=1.0,
)
output2 = tok2.batch_decode(out, skip_special_tokens=True)[0]


In [4]:
output2

'Yesterday, I visited a small bookstore near my apartment, and the owner recommended a novel about a young musician moving to Paris. Although I usually prefer science fiction, I decided to try it. I read the first three chapters while waiting for the bus.'

## Evaluation

In [5]:
import sacrebleu 

hyp = [output2]
ref = [[src]]

bleu = sacrebleu.corpus_bleu(hyp,ref)
chrf = sacrebleu.corpus_chrf(hyp,ref)
print(f"BLEU: {bleu.score:.1f}  chrF: {chrf.score:.1f}")



BLEU: 54.4  chrF: 74.7


# Ex 2. 

In [6]:
from langdetect import detect 

print(detect("Bonjour tout le monde"))
print(detect("Hello, how are you?"))


fr
en


In [7]:
import pycountry

def nllb_to_lang(nllb_code):
    code_3 = nllb_code.split("_")[0]
    lang = pycountry.languages.get(alpha_3=code_3) 
    return getattr(lang, "alpha_2", code_3)



In [8]:

tok = AutoTokenizer.from_pretrained(model_id, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
src = "Yesterday, I visited a small bookstore near my apartment. The owner recommended a novel about a young musician who moves to Paris. Although I usually prefer science fiction, I decided to give it a try. I read the first three chapters while waiting for the bus. By the time I got home, I was eager to find out what happened next."

def translate_with_lid(src=src, tgt_lang_nllb="fra_Latn", tok=tok, model=model):
     
    inputs = tok(src, return_tensors="pt")

    out = model.generate(
        **inputs, 
        forced_bos_token_id=tok.convert_tokens_to_ids(tgt_lang_nllb), 
        num_beams=5, 
        length_penalty=1.0,
    )
    candidate = tok.batch_decode(out, skip_special_tokens=True)[0]

    detected_lang = detect(candidate)
    expected_lang = nllb_to_lang(tgt_lang_nllb)


    if (expected_lang != detected_lang): 
        raise ValueError(f"Off-target translation detected: expected '{expected_lang}', but got '{detected_lang}'")
    return candidate

result = translate_with_lid(src, tgt_lang_nllb="fra_Latn")
print("Bản dịch thành công:", result)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Bản dịch thành công: Hier, j'ai visité une petite librairie près de mon appartement. Le propriétaire m'a recommandé un roman sur un jeune musicien qui déménage à Paris. Bien que je préfère généralement la science-fiction, j'ai décidé de l'essayer. J'ai lu les trois premiers chapitres en attendant le bus.


# Ex 3

In [9]:
from datasets import load_dataset, Dataset

# Helper functions
def flatten_fn(example): 
    return {
        "src": example["translation"]["en"],
        "tgt": example["translation"]["vi"]
    }

raw_ds = load_dataset("Helsinki-NLP/opus-100", "en-vi", split="train[:5000]") 
ds = raw_ds.map(flatten_fn, remove_columns=["translation"])

train_ds = ds[:4800]
test_ds = ds[4800:]

sources = test_ds['src']
references = [test_ds['tgt']]


In [10]:
hypotheses = []
for source in sources:
    inputs = tok(source, return_tensors="pt")

    out = model.generate(
        **inputs, 
        forced_bos_token_id=tok.convert_tokens_to_ids("vie_Latn"),
        num_beams=5,
        length_penalty=1.0,
    )

    output = tok.batch_decode(out, skip_special_tokens=True)[0]
    hypotheses.append(output)
    print(output)


Anh đã nói chuyện với họ chưa?
Thật khó để nói với một số người!
Nó đi vào DNA của bạn, vì vậy bạn chuyển vấn đề cho trẻ em của bạn.
Đó là lý do họ đề nghị cho tôi công việc, bởi vì tôi có thể liên kết với Avatar của Tommy.
Không phải lúc nào cũng vậy.
Ít nhất là lúc này.
Chúng ta sẽ làm thế.
Họ đang đuổi theo một máy tính xách tay mà anh ta để lại trong cab của chúng tôi.
Nhưng Philip, nó không phải là Robin.
Tôi không thể dễ dàng với anh.
Đây chỉ là một cái gì đó mà thôi.
Vậy, tôi có thể giúp gì được?
Vấn đề là sự lựa chọn.
Này, người lớn!
Bạn có nhận ra rằng cơ sở này thuộc sở hữu của vua Petyr Baelish, chủ của vua của Coin...
Giấy phép của anh à?
Chúng tôi cố gắng giữ nó càng gần với bản gốc càng tốt.
Cô ấy có thể xử lý được.
Những thiên thần khác, họ ghét anh.
Tôi nói anh ta gặp khó khăn.
- Có luật chống lại điều đó không?
Gào lên cho tôi.
Ừ, tôi biết nó ở đâu.
Được rồi...


KeyboardInterrupt: 

In [18]:
import sacrebleu

baseline_chrf = sacrebleu.corpus_chrf(hypotheses, references)
baseline_bleu = sacrebleu.corpus_bleu(hypotheses, references)
print(f"Baseline BLEU & chrF (before finetuning): {baseline_bleu.score:.2f} and {baseline_chrf.score:.2f}")


Baseline BLEU & chrF (before finetuning): 4.34 and 11.60


# Preprocessing


In [11]:
tok.src_lang ="eng_Latn"
tok.tgt_lang ="vie_Latn"

def preprocess(ex): 
    return tok(
        ex["src"],
        text_target=ex["tgt"],
        truncation=True, 
        max_length=128
    )

train_ds = Dataset.from_dict(train_ds)

train_tokenized = train_ds.map(preprocess, remove_columns=['src', 'tgt'])


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

In [12]:
train_tokenized[0]

{'input_ids': [256047, 7644, 248, 796, 248130, 2],
 'attention_mask': [1, 1, 1, 1, 1, 1],
 'labels': [256193, 176075, 9553, 4787, 248130, 2]}

# Finetuning


In [13]:
from transformers import Seq2SeqTrainer
from transformers import Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tok, model=model)


args = Seq2SeqTrainingArguments(
    output_dir="./nllb_finetuned",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=50,
    save_strategy="no",
    report_to="none")

trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_tokenized, data_collator=data_collator)
trainer.train()

/Users/quocanhngonguyen/Downloads/AI Engineering From Scratch/ai-engineering-from-scratch/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,1.967298
100,1.973011
150,1.862954
200,1.876745
250,2.054006
300,1.892177
350,1.700820
400,1.849865
450,1.879460
500,1.846495


TrainOutput(global_step=1200, training_loss=1.8474241447448732, metrics={'train_runtime': 1675.499, 'train_samples_per_second': 2.865, 'train_steps_per_second': 0.716, 'total_flos': 164945442078720.0, 'train_loss': 1.8474241447448732, 'epoch': 1.0})

# Step 5 - Re Evaluation

In [19]:
post_hypotheses= []
model.eval()
for source in sources: 
    inputs = tok(source, return_tensors="pt").to(model.device)

    out = model.generate(
        **inputs, 
        forced_bos_token_id=tok.convert_tokens_to_ids("vie_Latn"),
        num_beams=5,
        length_penalty=1.0,
    )

    output = tok.batch_decode(out, skip_special_tokens=True)[0]
    post_hypotheses.append(output)
    print(output)

Anh đã nói chuyện với họ chưa?
Thật khó để nói với một số người!
Nó vào trong DNA của anh, anh chuyển vấn đề cho con của anh.
Đó là lý do họ đề nghị cho tôi công việc, vì tôi có thể liên kết với Avatar của Tommy.
Tôi không phải lúc nào cũng làm vậy.
Ít nhất là lúc này.
Chúng ta.
Họ đang đuổi theo laptop mà hắn bỏ lại trong xe của chúng ta.
Nhưng Philip, không cần phải là Robin.
Tôi không thể dễ dàng với anh.
Đây chỉ là một cái gì đó mà thôi.
Vậy tôi có thể giúp gì được?
Vấn đề là sự lựa chọn.
Hey, người lớn!
Ông có nhận ra rằng nhà máy này thuộc sở hữu của ngài Petyr Baelish, chủ sưu tầm của vua...
Giấy phép của anh?
Chúng tôi cố giữ nó càng gần với bản gốc càng tốt.
Cô ấy có thể xử lý được.
Những thiên thần khác, họ ghét anh.
Tôi đã nói anh ta gặp khó khăn rồi.
- Có luật cấm không?
Cứ hét lên cho tôi.
Ừ, tôi biết nó ở đâu.
Được rồi...
Giờ thì chuyện gì vậy?
Thiết lập thư mục làm việc cho ứng dụng của bạn.
Và Killer Lemonade của Kyle.
Tôi không cần phải đến Denver.
Sao lại quan trọng t

In [20]:
post_bleu = sacrebleu.corpus_bleu(post_hypotheses, references)
post_chrf = sacrebleu.corpus_chrf(post_hypotheses, references)

print("=" * 45)
print(f"TRƯỚC Fine-tuning: BLEU = {baseline_bleu.score:.2f} | chrF = {baseline_chrf.score:.2f}")
print(f"SAU   Fine-tuning: BLEU = {post_bleu.score:.2f} | chrF = {post_chrf.score:.2f}")
print("=" * 45)

TRƯỚC Fine-tuning: BLEU = 4.34 | chrF = 11.60
SAU   Fine-tuning: BLEU = 22.05 | chrF = 39.41


In [22]:
import sacrebleu

deltas = []
for i in range(len(sources)):
    ref = test_ds['tgt'][i]
    b_hyp = hypotheses[i]
    a_hyp = post_hypotheses[i]
    
    # Tính điểm từng câu trước và sau
    score_before = sacrebleu.sentence_chrf(b_hyp, [ref]).score
    score_after = sacrebleu.sentence_chrf(a_hyp, [ref]).score
    diff = score_after - score_before
    
    deltas.append({
        "i": i,
        "diff": diff,
        "src": sources[i],
        "ref": ref,
        "before": b_hyp,
        "after": a_hyp
    })

# Sắp xếp để lấy Top 3 câu tiến bộ nhất và Top 3 câu bị thụt lùi
improved = sorted(deltas, key=lambda x: x["diff"], reverse=True)[:3]
regressed = sorted(deltas, key=lambda x: x["diff"])[:3]

print("--- TOP CÂU TIẾN BỘ NHẤT (IMPROVED) ---")
for item in improved:
    print(f"\n[Delta: +{item['diff']:.1f}]")
    print(f"EN   : {item['src']}")
    print(f"Mẫu  : {item['ref']}")
    print(f"Trước: {item['before']}")
    print(f"Sau  : {item['after']}")

print("\n" + "="*50 + "\n")

print("--- TOP CÂU BỊ THỤT LÙI (REGRESSED) ---")
for item in regressed:
    print(f"\n[Delta: {item['diff']:.1f}]")
    print(f"EN   : {item['src']}")
    print(f"Mẫu  : {item['ref']}")
    print(f"Trước: {item['before']}")
    print(f"Sau  : {item['after']}")


IndexError: list index out of range